# Backfill: Qwen2.5-7B, text mode

Runs **only** the one missing config — `Qwen/Qwen2.5-7B` in **text** scoring mode
(no permutation) on CUDA in bfloat16 — and saves it to
`/kaggle/working/results/qwen7b_text_cuda.csv` (+ its `_metrics.csv`).

**Before running:** *Settings* → **Accelerator = GPU** (16 GB T4/P100 is enough),
**Internet = On**. Optional `HF_TOKEN` Kaggle secret (Qwen2.5 is ungated).

In [ ]:
# 1. Clone the pipeline (ephemeral dir; keeps /kaggle/working output = CSVs only).
!rm -rf /tmp/bias-scaling
!git clone --depth 1 https://github.com/manitawtani74/bias-scaling.git /tmp/bias-scaling

In [ ]:
# 2. Deps. Kaggle already ships CUDA torch — do NOT reinstall it.
!pip install -q -U transformers datasets accelerate

In [ ]:
# 3. Optional HF token from a Kaggle Secret named HF_TOKEN.
import os
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
    print("HF token loaded from Kaggle secret.")
except Exception as e:
    print("No HF token (fine — Qwen2.5 is ungated):", e)

In [ ]:
# 4. Run the single missing config and save its two CSVs immediately.
import os, sys, subprocess, shutil
import torch
print("CUDA available:", torch.cuda.is_available())

REPO = "/tmp/bias-scaling"
WORK = "/tmp/results"            # eval writes here first
OUT = "/kaggle/working/results"  # persisted Kaggle output
os.makedirs(WORK, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

base = "qwen7b_text_cuda"
work_csv = f"{WORK}/{base}.csv"
cmd = [sys.executable, "-m", "src.evaluate",
       "--model", "Qwen/Qwen2.5-7B", "--device", "cuda", "--dtype", "bfloat16",
       "--sample", "200", "--seed", "0", "--output", work_csv]
print(">>>", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=REPO, check=True)

# Copy the run's two CSVs to the persisted output dir immediately.
for src in (work_csv, work_csv.replace(".csv", "_metrics.csv")):
    if os.path.exists(src):
        shutil.copy(src, OUT)
        print("saved ->", os.path.join(OUT, os.path.basename(src)), flush=True)